# Training ViTVS for Bird Sound Denoising
This notebook contains the training pipeline for the Vision Transformer for Vocal Separation (ViTVS) model. It is designed to run on Google Colab.

Make sure to set your Colab Runtime to **GPU** (Runtime -> Change runtime type -> Hardware accelerator -> GPU).

In [ ]:
# Clone the repository from GitHub
!git clone https://github.com/user312982/bird-denoising-sound.git
%cd bird-denoising-sound

# Install any necessary dependencies (e.g. torchaudio if needed)
!pip install torchaudio

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import os

# Import the model components
from models.vitvs import ViTVS

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Dataset Definition
Define a custom PyTorch Dataset for loading noisy bird sounds and their clean counterparts. You will need to upload your dataset to Colab or download it via script.

In [ ]:
class BirdDenoisingDataset(Dataset):
    def __init__(self, noisy_dir, clean_dir, target_length=32000):
        """
        Args:
            noisy_dir: Directory containing noisy audio files.
            clean_dir: Directory containing clean audio files.
            target_length: Fixed length for all audio clips (padding/truncating).
        """
        self.noisy_dir = noisy_dir
        self.clean_dir = clean_dir
        # Assuming files have the same names in both directories
        self.filenames = [f for f in os.listdir(noisy_dir) if f.endswith('.wav')]
        self.target_length = target_length
        
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        filename = self.filenames[idx]
        
        noisy_path = os.path.join(self.noisy_dir, filename)
        clean_path = os.path.join(self.clean_dir, filename)
        
        noisy_wav, sr = torchaudio.load(noisy_path)
        clean_wav, _ = torchaudio.load(clean_path)
        
        # Convert to mono if stereo
        if noisy_wav.shape[0] > 1:
            noisy_wav = torch.mean(noisy_wav, dim=0, keepdim=True)
        if clean_wav.shape[0] > 1:
            clean_wav = torch.mean(clean_wav, dim=0, keepdim=True)
            
        # Remove channel dim
        noisy_wav = noisy_wav.squeeze(0)
        clean_wav = clean_wav.squeeze(0)
        
        # Pad or truncate to target_length
        if noisy_wav.shape[0] > self.target_length:
            noisy_wav = noisy_wav[:self.target_length]
            clean_wav = clean_wav[:self.target_length]
        elif noisy_wav.shape[0] < self.target_length:
            pad_amount = self.target_length - noisy_wav.shape[0]
            noisy_wav = torch.nn.functional.pad(noisy_wav, (0, pad_amount))
            clean_wav = torch.nn.functional.pad(clean_wav, (0, pad_amount))
            
        return noisy_wav, clean_wav

# --- TODO: Update these paths to your actual dataset locations in Colab ---
# dataset = BirdDenoisingDataset(noisy_dir='/path/to/noisy', clean_dir='/path/to/clean', target_length=32000)
# dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2)

# For demonstration, we create a dummy dataloader
class DummyDataset(Dataset):
    def __len__(self): return 100
    def __getitem__(self, idx):
        return torch.randn(32000), torch.randn(32000)

dataloader = DataLoader(DummyDataset(), batch_size=8, shuffle=True)

## Training Setup

In [ ]:
# Initialize Model
# We use parameters suitable for 32000 sample audio with n_fft=1024, hop=256
model = ViTVS(
    img_size=(513, 126), # (n_fft//2 + 1, target_length // hop_length + 1)
    patch_size=16, 
    embed_dim=768, 
    depth=12, 
    num_heads=12, 
    num_classes=2,
    n_fft=1024, 
    hop_length=256, 
    win_length=1024
).to(device)

# NLL Loss requires log probabilities, which our model computes internally or we apply log_softmax
criterion = nn.NLLLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

epochs = 50

## Training Loop

In [ ]:
print("Starting Training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for i, (noisy_wav, clean_wav) in enumerate(dataloader):
        noisy_wav = noisy_wav.to(device)
        clean_wav = clean_wav.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        # denoised_wav: the reconstructed audio (B, T)
        # logits: spatial patch logits (B, NumClasses, Freq, Time)
        denoised_wav, logits = model(noisy_wav)
        
        # Apply log softmax for NLL Loss
        log_probs = torch.nn.functional.log_softmax(logits, dim=1)
        
        # Generate pseudo-labels for training the mask
        # In practice, you might train against ideal ratio masks (IRM) or binary masks.
        # Here, we generate a dummy label based on energy differences (just as a placeholder).
        # You should replace this with your actual target mask generation logic!
        B, C, F, T = logits.shape
        target_mask = torch.randint(0, 2, (B, F, T)).to(device) # Random dummy mask
        
        # Calculate loss
        loss = criterion(log_probs, target_mask)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

print("Training Complete!")

In [ ]:
# Save the trained model
torch.save(model.state_dict(), 'vitvs_bird_denoising.pth')
print("Model saved to vitvs_bird_denoising.pth")